# Olympus threat model walkthrough

This notebook is an interactive companion to `docs/01_threat_model.md`. It focuses on what Olympus **protects**, what it **does not protect**, and how to reason about common attack scenarios.


## Security properties and non-goals

Olympus is an append-only integrity system. It makes tampering, history rewriting, and inconsistent replication easier to detect, but it does not guarantee that every document was submitted or that operators made correct policy choices.


In [ ]:
protections = [
    'tamper evidence for committed records',
    'verifiable Merkle inclusion and chain linkage',
    'independent replay of canonicalization and hashing',
]

non_goals = [
    'forcing agencies to publish every document',
    'deciding what should be redacted',
    'replacing access control, legal review, or public records law',
]

for label, values in [('Protects', protections), ('Does not protect', non_goals)]:
    print(label)
    for item in values:
        print(f'  - {item}')
    print()


## Attack scenarios

We care about whether Olympus makes an attack **detectable**, not whether the attack is prevented from ever being attempted.


In [ ]:
attack_scenarios = [
    {
        'name': 'Silent document rewrite',
        'attacker': 'agency insider',
        'olympus_response': 'detectable',
        'why': 'canonical bytes and Merkle commitments change, so prior proofs stop matching',
    },
    {
        'name': 'Selective withholding before commitment',
        'attacker': 'records custodian',
        'olympus_response': 'not prevented',
        'why': 'Olympus proves what was committed, not what was omitted',
    },
    {
        'name': 'Forked shard history',
        'attacker': 'compromised node',
        'olympus_response': 'detectable',
        'why': 'header signatures, timestamps, and append-only linkage diverge',
    },
]

for scenario in attack_scenarios:
    print(f"{scenario['name']}: {scenario['olympus_response']}")
    print(f"  attacker: {scenario['attacker']}")
    print(f"  why: {scenario['why']}")
    print()


## Omission versus tampering

A missing document and a modified document are different failures. Olympus gives strong evidence about a committed document hash and its proofs, but non-disclosure still requires external accountability mechanisms.


In [ ]:
def classify_event(committed: bool, hash_matches: bool) -> str:
    if committed and hash_matches:
        return 'committed and intact'
    if committed and not hash_matches:
        return 'tampering is evident'
    return 'outside Olympus coverage (never committed or not yet observed)'

examples = [
    ('published version', True, True),
    ('rewritten version', True, False),
    ('withheld document', False, False),
]

for name, committed, matches in examples:
    print(f'{name}: {classify_event(committed, matches)}')


## How to use this notebook

1. Replace the sample attack scenarios with cases from your own records workflow.
2. Map each scenario to the Olympus pipeline: ingest → canonicalize → hash → commit → prove → verify.
3. Record which properties are guaranteed cryptographically and which still rely on institutional process.
